# Reconstruct: 1 イベント 1 行の再構成ツリー

TPC の反跳電子のエネルギー $K_e$(`TPC_Calibration`)と飛跡(`TOT_Skewness` の散乱点 $P_e$ と方向 $V_e$)、
シンチレータの散乱ガンマ線のエネルギー $E_g$・吸収点 $P_g$(`PSA_Event`)から、コンプトン散乱の量を求めて、`Reconstruct/single_hit` に 1 イベント 1 行で書きます。
式は SMILE_tool_box の `Erica/Reconstruct`(TRA、SingleHit)と SMLib の `Compton_common` と同じです。

| 書くイベント | シンチレータの 1 ヒット(`PSA_SingleHit`)と TPC のエネルギー(`TPC_Ee_Valid`)の両方があるもの。運動学が合わない($|\cos\alpha_{kin}| > 1$)ものは `NotConformEnergy` で除く |
|---|---|

| 列 | 式・中身 |
|---|---|
| `evt_id` | トリガーカウンタ(`ReadData`) |
| `Eg`, `Ke`, `E0` | $E_0 = E_g + K_e$ [keV] |
| `Pg[3]` | 吸収点 [mm] |
| `cos_phi` | 散乱角: $\cos\phi = 1 - m_ec^2 (1/E_g - 1/E_0)$ |
| `cos_psi_kin` | 反跳角: $\cos\psi = (1 + m_ec^2/E_0)\sqrt{K_e / (K_e + 2 m_ec^2)}$ |
| `cos_alpha_kin` | 散乱ガンマ線と反跳電子のなす角(運動学): $\cos\alpha_{kin} = (1 - m_ec^2/E_g)\sqrt{K_e / (K_e + 2 m_ec^2)}$ |
| `max_cos_alpha` | $k = E_0/m_ec^2$ として $\frac{k-1}{k+2}\sqrt{\frac{k^2-1}{k(k+2)}}$ |

`TOT_Skewness` がチェーンにあるとき(`smile3_reconstruct` にはある)は、飛跡を使った列も書きます。

| 列 | 式・中身 |
|---|---|
| `has_track` | 1: このイベントに `TOT_Ve_TRA`(飛跡の方向)がある。0 のとき下の列は 0xffffffff |
| `Pe[3]` | 散乱点(飛跡の始点、`TOT_Skewness` の `Pe`)[mm] |
| `Ve[3]` | 反跳電子の方向 $\hat e$(`TOT_Skewness` の `Ve` は長さに意味の無い方向なので、ここで単位ベクトルにする) |
| `Vg[3]` | 散乱ガンマ線の方向 $\hat g = (P_g - P_e)/|P_g - P_e|$ |
| `cos_alpha_geo` | 幾何の $\cos\alpha_{geo} = \hat g \cdot \hat e$ |
| `delta_cos_alpha` | $\cos\alpha_{geo} - \cos\alpha_{kin}$(正しく再構成できたイベントは 0 の近く) |
| `Vi[3]` | 入射ガンマ線の方向(線源から検出器へ): $(E_g \hat g + p_e \hat e)$ の単位ベクトル、$p_e = \sqrt{K_e (K_e + 2 m_ec^2)}$(`recon_vector`)。到来方向は $-V_i$ |
| `cos_arm`, `arm`, `spd` | `source_position` を書いたときだけ。ARM = $\angle(P_e - P_{src}, \hat g) - \phi$ [deg]、SPD = 2 つの面 $(P_e - P_{src}, \hat g)$ と $(V_i, \hat g)$ のなす角(符号つき)[deg] |

> **今の状態**: TPC のゲインマップが無いので $K_e$ は ADU × サンプルで、角度の値には意味がありません。
> また、TPC の位置(`TOT_Skewness`、SMILE-2+ のパネルの並びのまま μ-PIC の中心が原点)と、シンチレータのジオメトリ(`geometry.csv`、X: 東、Y: 南、Z: 下)が同じ座標系になっているかは、まだ確かめていません。

In [ ]:
# ===== 入力と出力(絶対パス。データ量が大きいので、リポジトリの外の NAS などに置く)=====
from pathlib import Path
# 入力: SMILE-3 FM のデータ(<日付>/calltt/<ラン>, <日付>/killua/<ラン>)
DATA = Path("/mnt/nas_eureka/SMILE-3FM_data")
# 出力: チェーンの出力の置き場(チュートリアル間で共有)
CACHE = Path("/mnt/nas_eureka/onoda/<書き換え>/EricaNext_tutorial")

for p in (DATA, CACHE):
    if "<" in str(p):
        raise ValueError(f"パスを書き換えてください: {p}")
CACHE.mkdir(parents=True, exist_ok=True)
print("入力:", DATA)
print("出力:", CACHE)


In [1]:
import os, sys, json, subprocess
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import ROOT
ROOT.gROOT.SetBatch(True)

# リポジトリの場所、チェーンの実行スクリプト、設定ファイル(config/*.json)の読み込み
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "run_script" / "run_chain.py").exists())
RUN_CHAIN = REPO / "run_script" / "run_chain.py"
sys.path.insert(0, str(REPO / "config"))
import ericanext_config as ec

def th2(h):
    """ROOT の TH2 -> (中身 [x のビン, y のビン], x のビンの端, y のビンの端)"""
    nx, ny = h.GetNbinsX(), h.GetNbinsY()
    dt = np.float32 if h.InheritsFrom("TArrayF") else np.float64   # TH2F / TH2D
    a = np.frombuffer(h.GetArray(), dtype=dt, count=(nx + 2) * (ny + 2)).reshape(ny + 2, nx + 2).astype(float)
    ex = np.array([h.GetXaxis().GetBinLowEdge(i) for i in range(1, nx + 2)])
    ey = np.array([h.GetYaxis().GetBinLowEdge(i) for i in range(1, ny + 2)])
    return a[1:-1, 1:-1].T.copy(), ex, ey

def th1(h):
    """ROOT の TH1 -> (中身, ビンの中心)"""
    n = h.GetNbinsX()
    return np.array([h.GetBinContent(i) for i in range(1, n + 1)]), np.array([h.GetBinCenter(i) for i in range(1, n + 1)])

def vec(column):
    """RDataFrame.AsNumpy() の配列型の列 -> numpy の配列のリスト"""
    return [np.asarray(v) for v in column]

def show_params(config, module):
    """設定ファイルの、あるモジュールのパラメータを、説明(_doc)と単位(_units)つきで表示する"""
    p = json.load(open(REPO / "config" / f"{config}.json"))["parameters"][module]
    doc, units = p.get("_doc", {}), p.get("_units", {})
    for k, v in p.items():
        if k.startswith("_"):
            continue
        u = f" [{units[k]}]" if k in units else ""
        print(f"{k} = {json.dumps(v, ensure_ascii=False)}{u}\n    {doc.get(k, '')}")

## 1. データとチェーン

TPC とシンチレータを一緒に読むので、ETCC のラン `20261102_run04`(NAS、211 イベント)を、2 つの storage のディレクトリ(`--data-dir` と `--extra-data-dir`)から流します。
設定ファイルは `smile3_reconstruct`(`… → PSA_Event → TOT_Skewness → Reconstruct`)で、シンチレータのエネルギーとジオメトリは `calib/psa` のものを使います。

In [2]:
RUN = "20261102_run04"
RUN_DIR, EXTRA_DIR = DATA / "20261102" / "calltt" / RUN, DATA / "20261102" / "killua" / RUN
OUT = CACHE / f"reconstruct_{RUN}.root"
cmd = [sys.executable, str(RUN_CHAIN), "smile3_reconstruct", "-o", str(OUT), "--data-dir", str(RUN_DIR), "--extra-data-dir", str(EXTRA_DIR)]
print(" ".join(cmd))
r = subprocess.run(cmd, capture_output=True, text=True); r.check_returncode()
for l in r.stdout.splitlines():
    if l.split()[:1] and l.split()[0] in ("PSA_SingleHit", "PSA_MultiHit", "PSA_NoHits", "TPC_Ee_Valid", "TOT_Ve_TRA", "NotConformEnergy"):
        print(l)

/usr/bin/python3 /home/onoda/ANLNext/Erica_git_dev/EricaNext/run_script/run_chain.py smile3_reconstruct -o /home/onoda/ANLNext/Erica_git_dev/EricaNext/tutorials/cache/reconstruct_20261102_run04.root --data-dir /mnt/nas_eureka/SMILE-3FM_data/20261102/calltt/20261102_run04 --extra-data-dir /mnt/nas_eureka/SMILE-3FM_data/20261102/killua/20261102_run04


NotConformEnergy                                            7                0
PSA_MultiHit                                              138              136
PSA_NoHits                                                 10               10
PSA_SingleHit                                              63               56
TOT_Ve_TRA                                                189              183
TPC_Ee_Valid                                              202              195


## 2. single_hit

式どおりに入っているかを、ツリーの値から計算し直して確かめます($K_e$ が ADU なので、値そのものには意味がありません)。

In [3]:
cols = ["evt_id", "Eg", "Ke", "Pg", "cos_phi", "cos_alpha_kin", "has_track", "Pe", "Ve", "Vg", "cos_alpha_geo", "Vi"]
s = ROOT.RDataFrame("Reconstruct/single_hit", str(OUT)).AsNumpy(cols)
print(len(s["evt_id"]), "rows,", int(s["has_track"].sum()), "with the track")
me = 510.99895
for i in range(min(5, len(s["evt_id"]))):
    eg, ke = s["Eg"][i], s["Ke"][i]; e0 = eg + ke
    line = (f"evt_id {s['evt_id'][i]:4d}: Eg {eg:7.1f} Ke {ke:8.1f}  cos_phi {s['cos_phi'][i]:+.3f} (= {1 - me * (1 / eg - 1 / e0):+.3f})"
            f"  cos_alpha_kin {s['cos_alpha_kin'][i]:+.3f}")
    if s["has_track"][i]:
        pg, pe, ve = np.array(s["Pg"][i]), np.array(s["Pe"][i]), np.array(s["Ve"][i])
        g = (pg - pe) / np.linalg.norm(pg - pe)
        vi = eg * g + np.sqrt(ke * (ke + 2 * me)) * ve
        line += (f"  cos_alpha_geo {s['cos_alpha_geo'][i]:+.3f} (= {g @ ve:+.3f})"
                 f"  Vi {np.round(np.array(s['Vi'][i]), 3)} (= {np.round(vi / np.linalg.norm(vi), 3)})")
    print(line)

53 rows, 50 with the track
evt_id    2: Eg   338.7 Ke   1277.2  cos_phi -0.193 (= -0.193)  cos_alpha_kin -0.379  cos_alpha_geo +0.772 (= +0.772)  Vi [-0.028 -0.994  0.105] (= [-0.028 -0.994  0.105])
evt_id    3: Eg  1348.3 Ke   1183.7  cos_phi +0.823 (= +0.823)  cos_alpha_kin +0.455  cos_alpha_geo -0.664 (= -0.664)  Vi [-0.477  0.582  0.659] (= [-0.477  0.582  0.659])
evt_id    4: Eg  1370.6 Ke   3330.9  cos_phi +0.736 (= +0.736)  cos_alpha_kin +0.549  cos_alpha_geo +0.092 (= +0.092)  Vi [0.148 0.945 0.293] (= [0.148 0.945 0.293])
evt_id    5: Eg   739.8 Ke   5672.6  cos_phi +0.389 (= +0.389)  cos_alpha_kin +0.285  cos_alpha_geo +0.720 (= +0.720)  Vi [0.181 0.819 0.545] (= [0.181 0.819 0.545])
evt_id    7: Eg   257.5 Ke     41.6  cos_phi +0.724 (= +0.724)  cos_alpha_kin -0.195


## 3. パラメータ

| パラメータ | 中身 |
|---|---|
| `source_position` | 線源の位置 [mm](検出器の座標系、3 つの値)。書くと `cos_arm`、`arm`、`spd` を出す。空(既定)なら出さない |

$m_ec^2 = 510.99895$ keV です。